In [ ]:
%%capture cap
%run ./src/desp-authentication.py

In [2]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]
access_token

'Token successfully written to /home/koenifra/.polytopeapirc'

In [3]:
import pandas as pd
import concurrent.futures
import xarray as xr
import os

In [4]:
geosphere_stations = pd.read_csv("./points/points_IFS_NEMO_ScenarioMIP_141.csv")

In [5]:
from typing import Literal
def extract_cdt_ts(experiment: str,
                   activity: str,
                   level_type: str,
                   datestring: str,
                   model: str,
                   parameter: str,
                   location: list,
                   feature: Literal["timeseries", "polygon"]="timeseries",
                   time_resolution: str="0000/to/2300",
                   resolution: str="high"):
    import earthkit.data
    
    if feature == "timeseries":
        feature_dict = {
            "type" : "timeseries",
            "points": location,
            "time_axis": "date"
        }
    elif feature == "polygon":
        feature_dict = {
            "type" : "polygon",
            "shape": location
        }
    else:
        raise TypeError("feature not supported")
    
    request = {
        # static parameters of climate dt data
        "class": "d1",
        "dataset": "climate-dt",
        "generation": "1",
        "expver": "0001",
        "stream": "clte",
        "type": "fc",
        # generic
        "activity": activity,
        "experiment": experiment,
        "levtype": level_type,
        "date": datestring,
        "model": model,
        "param": parameter,
        # "param": "167/228",
        "param": "141",
        "realization": "1",
        "resolution": resolution,
        "time": time_resolution,
        "feature": feature_dict
    }

    # commented out to check if only one level is request it gets faster or not
    if level_type == "sol":
        # request["levelist"] = "1/to/5"
        request["levelist"] = "1"
    
    try:
        ds = earthkit.data.from_source("polytope", 
                                       "destination-earth", 
                                       request, stream=False, 
                                       address='polytope.lumi.apps.dte.destination-earth.eu')
        return ds.to_xarray()
    except Exception as e:
        print(e)
        return None
    # return ds.to_xarray()
    
    

In [6]:
def get_datestring(temp_extent: str):
    '''
    Example output string "20200101/to/20210101"
    '''
    years = temp_extent.split("-")
    return f"{years[0]}0101/to/{years[1]}1231"
    

In [14]:
experiment="SSP3-7.0"
activity="ScenarioMIP"
level_type="sfc"
datestring="2020-2024"
model="IFS-NEMO"
parameter="141"

In [15]:
def get_station_data(geosphere_station):
    latlon = [[float(geosphere_station["latitude"]), float(geosphere_station["longitude"])]]

    ts = extract_cdt_ts(
        experiment=experiment,
        activity=activity,
        level_type=level_type,
        datestring=get_datestring(datestring),
        model=model,
        parameter=parameter,
        location=latlon
    )

    if ts is None:
        return None

    point_id = int(geosphere_station["points"])

    return ts.stack(pointid=("latitude", "longitude")) \
             .reset_index("pointid") \
             .assign_coords({"stationid": ("pointid", [geosphere_station["points"]])}) \
             .assign_coords({"pointid": [point_id]}) \
             .transpose("pointid", ...)

In [ ]:
store_path = f"../dt_climate-zarr-save/{model}_{level_type}_{activity}_{experiment}testiiii2.zarr"
print(f"Data will be stored in: {store_path}")

Data will be stored in: ../dt_climate-zarr-save/IFS-NEMO_sfc_ScenarioMIP_SSP3-7.0testiiii2.zarr


In [18]:
with concurrent.futures.ThreadPoolExecutor(max_workers=25) as executor:
    futures = []
    for ind, station in geosphere_stations.iloc[:10].iterrows():
        futures.append(executor.submit(get_station_data, geosphere_station=station))
    
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        if result is None:
            print("Skipping failed station")
            continue
        if not os.path.exists(store_path):
            result.chunk(chunks={
                "pointid": 1,
                "levelist": 1,
                "number": 1,
                "datetime": 1,
                "t": "auto"
            }).to_zarr(store=store_path, mode="w")
        else:
            result.chunk(chunks={
                "pointid": 1,
                "levelist": 1,
                "number": 1,
                "datetime": 1,
                "t": "auto"
            }).to_zarr(store=store_path, append_dim="pointid")

2026-04-29 14:04:07 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-04-29 14:04:07 - INFO - Key read from /home/koenifra/.polytopeapirc
2026-04-29 14:04:07 - INFO - Sending request...
{'request': 'activity: ScenarioMIP\n'
            'class: d1\n'
            'dataset: climate-dt\n'
            'date: 20200101/to/20241231\n'
            'experiment: SSP3-7.0\n'
            "expver: '0001'\n"
            'feature:\n'
            '  points:\n'
            '  - - 43.008633994237\n'
            '    - 4.005\n'
            '  time_axis: date\n'
            '  type: timeseries\n'
            "generation: '1'\n"
            'levtype: sfc\n'
            'model: IFS-NEMO\n'
            "param: '141'\n"
            "realization: '1'\n"
            'resolution: high\n'
            'stream: clte\n'
            'time: 0000/to/2300\n'
            'type: fc\n',
 'verb': 'retrieve'}
2026-04-29 14:04:07 - INFO - Sending request...
{'request': 'activity: ScenarioMIP\n'
            'class: d1\n

Polytope error
Situation: trying to download data
Description: HTTP CLIENT ERROR (400)
URL: https://polytope.lumi.apps.dte.destination-earth.eu:443/api/v1/requests/6e69db9c-5dd6-4ad3-b44b-d183ef9ac81e
HTTP method: GET
Request header/body contents:
{'headers': {'Authorization': 'Bearer **********JyAw'}, 'json': None}
Expected responses: 200, 202
Received response: CLIENT ERROR (400)
Details:
Request failed with error:
Matched datasource polytope-climate

An error occurred (InternalError) when calling the UploadPart operation (reached max retries: 4): We encountered an internal error, please try again.
Skipping failed station


2026-04-29 14:04:39 - INFO - The current status of the request is 'processing'
2026-04-29 14:04:43 - INFO - The current status of the request is 'processing'


Polytope error
Situation: trying to download data
Description: HTTP CLIENT ERROR (400)
URL: https://polytope.lumi.apps.dte.destination-earth.eu:443/api/v1/requests/5a716a05-9041-4ae6-9447-754a7593f3e2
HTTP method: GET
Request header/body contents:
{'headers': {'Authorization': 'Bearer **********JyAw'}, 'json': None}
Expected responses: 200, 202
Received response: CLIENT ERROR (400)
Details:
Request failed with error:
Matched datasource polytope-climate

An error occurred (InternalError) when calling the UploadPart operation (reached max retries: 4): We encountered an internal error, please try again.
Skipping failed station


2026-04-29 14:05:10 - INFO - The current status of the request is 'processing'


...

Polytope error
Situation: trying to download data
Description: HTTP CLIENT ERROR (400)
URL: https://polytope.lumi.apps.dte.destination-earth.eu:443/api/v1/requests/5d0d4e77-783b-4f4e-876b-6033cd298693
HTTP method: GET
Request header/body contents:
{'headers': {'Authorization': 'Bearer **********JyAw'}, 'json': None}
Expected responses: 200, 202
Received response: CLIENT ERROR (400)
Details:
Request failed with error:
Matched datasource polytope-climate

An error occurred (InternalError) when calling the UploadPart operation (reached max retries: 4): We encountered an internal error, please try again.
Skipping failed station


2026-04-29 14:05:49 - INFO - The current status of the request is 'processing'
2026-04-29 14:06:08 - INFO - The current status of the request is 'processing'
2026-04-29 14:06:09 - INFO - The current status of the request is 'processing'
2026-04-29 14:06:10 - INFO - The current status of the request is 'processed'


Polytope error
Situation: trying to download data
Description: HTTP CLIENT ERROR (400)
URL: https://polytope.lumi.apps.dte.destination-earth.eu:443/api/v1/requests/cc753aa3-74fc-4ab4-93f2-701d98919b07
HTTP method: GET
Request header/body contents:
{'headers': {'Authorization': 'Bearer **********JyAw'}, 'json': None}
Expected responses: 200, 202
Received response: CLIENT ERROR (400)
Details:
Request failed with error:
Matched datasource polytope-climate

An error occurred (InternalError) when calling the UploadPart operation (reached max retries: 4): We encountered an internal error, please try again.
Skipping failed station


/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=20, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/koenifra/Projects/dt-climate-zarr/dt/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
2026-04-29 14:06:40 - INFO - The current st

Polytope error
Situation: trying to download data
Description: HTTP CLIENT ERROR (400)
URL: https://polytope.lumi.apps.dte.destination-earth.eu:443/api/v1/requests/16763546-4ad0-4504-b77e-85b1a24e6234
HTTP method: GET
Request header/body contents:
{'headers': {'Authorization': 'Bearer **********JyAw'}, 'json': None}
Expected responses: 200, 202
Received response: CLIENT ERROR (400)
Details:
Request failed with error:
Matched datasource polytope-climate

An error occurred (InternalError) when calling the UploadPart operation (reached max retries: 4): We encountered an internal error, please try again.
Skipping failed station
